In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/Housing.csv")
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
df.shape

(545, 13)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [6]:
X = df.drop(columns=['airconditioning'])
y = df['airconditioning']

In [7]:
cat_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'prefarea']

for col in cat_cols:
    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])

In [8]:
encoder_y = LabelEncoder()
y = encoder_y.fit_transform(y)

In [9]:
X = pd.get_dummies(X, columns=["furnishingstatus"], prefix="furnishingstatus", drop_first=True, dtype=int)

X.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,2,1,0,0
1,12250000,8960,4,4,4,1,0,0,0,3,0,0,0
2,12250000,9960,3,2,2,1,0,1,0,2,1,1,0
3,12215000,7500,4,2,2,1,0,1,0,3,1,0,0
4,11410000,7420,4,1,2,1,1,1,0,2,0,0,0


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.20, random_state=42
)

In [11]:
num_cols = ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [12]:
X_train.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
120,0.617469,0.655716,0.061047,-0.562857,-0.934237,1,1,1,0,1.467701,1,0,0
274,-0.243478,0.613133,1.461920,-0.562857,0.253306,1,0,0,0,-0.846648,0,1,0
268,-0.221018,-0.096581,1.461920,-0.562857,0.253306,1,0,0,0,-0.846648,0,1,0
517,-1.254153,-1.019208,-1.339825,-0.562857,-0.934237,1,0,0,0,0.310526,0,0,1
10,2.676253,3.806844,0.061047,-0.562857,0.253306,1,0,1,0,1.467701,1,0,0


In [13]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [14]:
train_score = model.score(X_train, y_train)
print("Train score : ", train_score)

test_score = model.score(X_test, y_test)
print("Test score : ", test_score)

Train score :  0.7775229357798165
Test score :  0.7522935779816514


In [15]:
cv5_score = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

print("Cv 5 scores : ", cv5_score)

Cv 5 scores :  [0.81818182 0.74712644 0.70114943 0.77011494 0.75862069]


In [16]:
cv10_score = cross_val_score(model, X_train, y_train, cv=10, scoring="accuracy")

print("Cv 10 scores : ", cv10_score)

Cv 10 scores :  [0.84090909 0.84090909 0.75       0.72727273 0.65909091 0.75
 0.74418605 0.8372093  0.81395349 0.76744186]
